In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine,text

import polars as pl
import plotly.express as px
import os, sys

module_path = os.path.abspath(os.path.join('../../../..')) # or the path to your source code
sys.path.insert(0, module_path)
import util

config = util.summary_settings
input_config = util.input_settings

In [2]:
hh = util.get_hh_data(quantile_groups=['hhincome','emptot_1','hh_1']).to_pandas()

In [3]:
df_acs = util.read_sqlite_db("SELECT * FROM observed_acs_vehicles_drivers").to_pandas()

df_acs = df_acs.groupby('vehicles').sum()[['households']].reset_index()
df_acs.rename(columns={'households': 'hhexpfac'}, inplace=True)
df_acs['source'] = 'ACS'
df_acs.replace(4, '4+', inplace=True)
df_acs.rename(columns={'vehicles':'hhvehs_4+'}, inplace=True)

In [8]:
df_plot = hh.groupby(['source','hhvehs_4+'])['hhexpfac'].sum().reset_index()
df_plot = pd.concat([df_plot,df_acs])
df_plot = df_plot.reset_index(drop=True)
df_plot['percentage'] = df_plot.groupby(['source'], group_keys=False)['hhexpfac'].\
        apply(lambda x: x / float(x.sum()))

df_plot_ct = hh.groupby(['source','hhvehs_4+'])['hhexpfac'].count().reset_index(). \
    rename(columns={'hhexpfac':'sample count'})
df_plot = df_plot.merge(df_plot_ct, on=['source','hhvehs_4+'], how='left')
df_plot = df_plot[df_plot['hhvehs_4+'] != '-1']

fig = px.bar(df_plot.sort_values(by=['source']), x="hhvehs_4+", y="percentage", color="source",
             hover_data=['sample count'],
             category_orders={'source': ['model','survey','ACS']},
             barmode="group",title="Auto ownership")
fig.update_layout(height=400, width=700, font=dict(size=11),
                  yaxis=dict(tickformat=".2%"))
fig.show()

## Auto ownership by segments

In [ ]:
# auto ownership in Income groups
def plot_auto(df:pd.DataFrame, var:str, title_cat:str, sub_name:str):
    df2 = df.loc[df['hhvehs_4+'] != '-1'].copy()
    df_plot = df2.groupby(['source',var,'hhvehs_4+'])['hhexpfac'].sum().reset_index()
    df_plot['percentage'] = df_plot.groupby(['source',var], group_keys=False)['hhexpfac'].\
        apply(lambda x: x / float(x.sum()))

    df_plot_ct = df2.groupby(['source',var,'hhvehs_4+'])['hhexpfac'].count().reset_index(). \
        rename(columns={'hhexpfac':'sample count'})
    df_plot = df_plot.merge(df_plot_ct, on=['source',var,'hhvehs_4+'])

    fig = px.bar(df_plot, x="hhvehs_4+", y="percentage", color="source",
                 facet_col=var, barmode="group",
                 hover_data=['sample count'],
                 title="Auto ownership by "+ title_cat)
    fig.for_each_annotation(lambda a: a.update(text = sub_name + "=<br>" + a.text.split("=")[-1]))
    fig.update_xaxes(title_text="n of cars")
    fig.update_layout(height=400, width=800, font=dict(size=11),
                      yaxis=dict(tickformat=".2%"))
    fig.for_each_yaxis(lambda a: a.update(tickformat = ".2%"))
    fig.show()

In [6]:
plot_auto(hh,'hhincome_4group','income level', 'Income')

In [7]:
plot_auto(hh,'hhsize_4+','household size', 'HH size')

In [8]:
plot_auto(hh.loc[hh['drivers_4+']!="0"],'drivers_4+','number of (poential) drivers age 16+','num drivers')

In [9]:
plot_auto(hh,'hhwkrs_4+','number of workers','num workers')

In [10]:
plot_auto(hh.dropna(subset=['hh_1_4group']),'hh_1_4group','household density','density')

In [11]:
plot_auto(hh.dropna(subset=['emptot_1_4group']),'emptot_1_4group','employment density','density')

## Validate auto ownership with ACS vehicle ownership data

In [12]:
df_acs

,hhvehs_4+,hhexpfac,source
0,0,143723.838187,ACS
1,1,583994.620470,ACS
2,2,628318.617840,ACS
3,3,250817.549140,ACS
4,4+,129274.374393,ACS


In [13]:
# ACS auto ownership validation dataset
df_acs = util.read_sqlite_db("SELECT * FROM bg_auto_ownership").to_pandas()
maz_bg_lookup = util.read_sqlite_db("SELECT * FROM maz_bg_lookup").to_pandas()

# add lookup for maz and block groups
df_acs = df_acs.merge(maz_bg_lookup, on='block_group_id')
df_acs_taz = df_acs[['TAZ','block_group_id','cars_none_control', 'cars_one_control','cars_two_or_more_control']].drop_duplicates()

hh_taz = hh.merge(df_acs_taz, how='left', left_on='hhtaz', right_on='TAZ')
